# RAG Answer Evaluation Pipeline with Advanced LLM-as-Judge Metrics

## Overview
This notebook implements production-grade evaluation using:
1. **NLI-Based Claim Deconstruction** - Break answer into micro-claims, evaluate against context
2. **Chain-of-Thought (CoT) Reasoning** - Force judges to reason before scoring
3. **Citation & Source Correctness** - Verify all claims are backed by exact chunks
4. **Multi-Step DAG-Based Metrics** - Deterministic scoring rubric

**Output Metrics Only:**
- Faithfulness
- Hallucination
- Relevance
- Coverage
- ELI15 Score
- Jargon Free Score
- Citation & Source Correctness
- Reason (Chain-of-Thought)


## 1. Install Dependencies


In [ ]:
!pip install pymupdf faiss-cpu sentence-transformers google-generativeai openpyxl pandas tqdm textstat rank-bm25 scikit-learn -q


## 2. Imports


In [ ]:
import os
import re
import json
import time
import uuid
import numpy as np
import pandas as pd
import fitz
import faiss
import textstat
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
import google.generativeai as genai
from google.colab import files
import openpyxl
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter
from rank_bm25 import BM25Okapi
import string
from collections import defaultdict

print("All imports successful.")


## 3. Configure Gemini API


In [ ]:
GEMINI_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your actual key
JUDGE_MODEL_NAME = "gemini-2.5-flash"  # LLM-as-Judge model

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel(model_name=JUDGE_MODEL_NAME)
print(f"Gemini judge model initialised: {JUDGE_MODEL_NAME}")


## 4. Upload Source PDFs


In [ ]:
uploaded_files = files.upload()
pdf_paths = list(uploaded_files.keys())

if not pdf_paths:
    raise ValueError("No PDFs uploaded")

print(f"Uploaded {len(pdf_paths)} PDF(s):")
for p in pdf_paths:
    print(f"  {p}")


## 5. Upload Question-Answer Pairs


In [ ]:
uploaded_qa = files.upload()
excel_path = list(uploaded_qa.keys())[0]

df_raw = pd.read_excel(excel_path, header=None)
df_qa = df_raw.iloc[1:, [2, 3]].copy()
df_qa.columns = ["Question", "LLM_Output"]

df_qa = df_qa.dropna(subset=["Question", "LLM_Output"])
df_qa["Question"] = df_qa["Question"].astype(str).str.strip()
df_qa["LLM_Output"] = df_qa["LLM_Output"].astype(str).str.strip()
df_qa = df_qa[(df_qa["Question"] != "") & (df_qa["LLM_Output"] != "")]
df_qa = df_qa.reset_index(drop=True)

print(f"Loaded {len(df_qa)} question-answer pairs.")
print()
print(df_qa.head(3))


## 6. Structure-Aware Parent-Child Chunking


In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract text from PDF with position awareness."""
    doc = fitz.open(pdf_path)
    text = ""
    for page_num in range(len(doc)):
        text += f"\n[Page {page_num + 1}]\n"
        text += doc[page_num].get_text()
    doc.close()
    return text


class HierarchicalChunker:
    """Split PDFs into hierarchical parent-child chunks based on document structure."""

    def __init__(self, parent_target_tokens=1000, child_target_tokens=200):
        self.parent_target = parent_target_tokens
        self.child_target = child_target_tokens
        self.chunks = []

    def split_by_headers(self, text):
        """Split text into sections based on markdown-like headers."""
        lines = text.split('\n')
        sections = []
        current_section = []
        current_header = None
        current_level = 999

        for line in lines:
            header_match = re.match(r'^(#{1,6})\s+(.+)$', line)
            if header_match:
                if current_section or current_header:
                    section_text = '\n'.join(current_section).strip()
                    if section_text or current_header:
                        sections.append({
                            'level': current_level,
                            'header': current_header,
                            'text': section_text
                        })
                current_level = len(header_match.group(1))
                current_header = header_match.group(2).strip()
                current_section = []
            else:
                if line.strip() or current_section:
                    current_section.append(line)

        if current_section or current_header:
            section_text = '\n'.join(current_section).strip()
            if section_text or current_header:
                sections.append({
                    'level': current_level,
                    'header': current_header,
                    'text': section_text
                })

        return sections

    def chunk_document(self, text, source_name):
        """Create hierarchical chunks: parents and children."""
        sections = self.split_by_headers(text)

        for section in sections:
            # Create PARENT chunk
            parent_id = str(uuid.uuid4())
            parent_text = f"[Policy: {source_name}] [Section: {section['header']}]\n\n{section['text']}"

            parent_chunk = {
                'id': parent_id,
                'text': parent_text,
                'parent_id': None,
                'level': section['level'],
                'source': source_name,
                'header': section['header'],
                'type': 'parent'
            }

            self.chunks.append(parent_chunk)

            # Create CHILD chunks from subsections
            parent_content_lines = section['text'].split('\n')
            child_sections = []
            current_child = []
            child_header = None

            for line in parent_content_lines:
                sub_match = re.match(r'^(#{' + str(section['level'] + 1) + r',})\s+(.+)$', line)
                if sub_match:
                    if current_child or child_header:
                        child_sections.append({
                            'header': child_header,
                            'text': '\n'.join(current_child).strip()
                        })
                    child_header = sub_match.group(2).strip()
                    current_child = []
                else:
                    current_child.append(line)

            if current_child or child_header:
                child_sections.append({
                    'header': child_header,
                    'text': '\n'.join(current_child).strip()
                })

            # Create child chunks
            for child_section in child_sections:
                if child_section['text'].strip():
                    child_id = str(uuid.uuid4())
                    child_text = f"[Policy: {source_name}] [Section: {section['header']}] [Subsection: {child_section['header']}]\n\n{child_section['text']}"

                    child_chunk = {
                        'id': child_id,
                        'text': child_text,
                        'parent_id': parent_id,
                        'level': section['level'] + 1,
                        'source': source_name,
                        'header': f"{section['header']} > {child_section['header']}",
                        'type': 'child'
                    }

                    self.chunks.append(child_chunk)

        return self.chunks


# Build hierarchical index
chunker = HierarchicalChunker(parent_target_tokens=1000, child_target_tokens=200)
all_chunks = []
chunk_metadata = {}
parent_chunks_dict = {}

for pdf_file in pdf_paths:
    raw_text = extract_text_from_pdf(pdf_file)
    source_name = pdf_file.replace('.pdf', '').replace('(1)', '').strip()
    chunks = chunker.chunk_document(raw_text, source_name)
    print(f"{pdf_file}")
    print(f"  Parents: {sum(1 for c in chunks if c['type'] == 'parent')}")
    print(f"  Children: {sum(1 for c in chunks if c['type'] == 'child')}")

# Extract child chunks for indexing and parent chunks for swapping
child_chunks_list = [c for c in chunker.chunks if c['type'] == 'child']
parent_chunks_dict = {c['id']: c for c in chunker.chunks if c['type'] == 'parent'}

for chunk in child_chunks_list:
    all_chunks.append(chunk['text'])
    chunk_metadata[len(all_chunks) - 1] = chunk

print(f"\nTotal child chunks (indexed): {len(all_chunks)}")
print(f"Total parent chunks (for swapping): {len(parent_chunks_dict)}")


## 7. Build FAISS Dense Index + BM25 Sparse Index (Child Chunks Only)


In [ ]:
# Load embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded.")

# Encode CHILD chunks only
chunk_embeddings = embedding_model.encode(
    all_chunks,
    batch_size=64,
    convert_to_numpy=True,
    show_progress_bar=True
)

faiss.normalize_L2(chunk_embeddings)

# Build FAISS index
embedding_dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(chunk_embeddings)

print(f"FAISS index ready. Dimension: {embedding_dim}, Child chunks: {faiss_index.ntotal}")

# Build BM25 index
_punct_table = str.maketrans('', '', string.punctuation)

def tokenise(text):
    return text.lower().translate(_punct_table).split()

tokenised_chunks = [tokenise(c) for c in all_chunks]
bm25_index = BM25Okapi(tokenised_chunks)

print(f"BM25 index ready. Corpus: {len(tokenised_chunks)} child chunks")


## 8. Hybrid Retrieval + Parent Swap


In [ ]:
TOP_K_FACTUAL = 5
POOL_K = 20
RRF_K = 60

QUERY_EXPANSIONS = {
    "vesting age": ["retirement age", "maturity age"],
    "surrender value": ["policy surrender", "withdrawal value"],
    "premium": ["premium payment", "premium amount"],
    "rider": ["add-on rider", "optional rider"],
    "waiting period": ["waiting period", "survival period"],
    "benefit": ["sum assured", "death benefit"],
}

try:
    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    print("Cross encoder loaded.")
except Exception as e:
    cross_encoder = None
    print(f"Cross encoder unavailable: {e}")


def generate_query_variants(query):
    """Query expansion for insurance concepts."""
    query_lower = query.lower()
    variants = [query]
    for trigger, expansions in QUERY_EXPANSIONS.items():
        if trigger in query_lower:
            variants.extend(expansions)
    seen = set()
    unique = []
    for v in variants:
        k = v.strip().lower()
        if k and k not in seen:
            seen.add(k)
            unique.append(v.strip())
    return unique


def retrieve_hybrid_single(query, pool=POOL_K, rrf_k=RRF_K):
    """Dense + Sparse hybrid retrieval with RRF on child chunks."""
    q_emb = embedding_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    dense_scores, dense_idxs = faiss_index.search(q_emb, pool)
    dense_ranks = {int(idx): rank for rank, idx in enumerate(dense_idxs[0]) if idx >= 0}

    tokens = tokenise(query)
    bm25_scores = bm25_index.get_scores(tokens)
    bm25_top = np.argsort(bm25_scores)[::-1][:pool]
    bm25_ranks = {int(idx): rank for rank, idx in enumerate(bm25_top)}

    all_idxs = set(dense_ranks) | set(bm25_ranks)
    rrf_scores = {}
    for idx in all_idxs:
        score = 0.0
        if idx in dense_ranks:
            score += 1.0 / (rrf_k + dense_ranks[idx] + 1)
        if idx in bm25_ranks:
            score += 1.0 / (rrf_k + bm25_ranks[idx] + 1)
        rrf_scores[idx] = score

    return [(idx, rrf_scores[idx]) for idx in sorted(rrf_scores, key=rrf_scores.get, reverse=True)]


def retrieve_multi_query(query, k=TOP_K_FACTUAL, pool=POOL_K):
    """Multi-query retrieval with merged scores."""
    variants = generate_query_variants(query)
    merged_scores = defaultdict(float)
    for variant in variants:
        for idx, score in retrieve_hybrid_single(variant, pool=pool):
            merged_scores[idx] = max(merged_scores[idx], score)
    ranked = sorted(merged_scores, key=merged_scores.get, reverse=True)
    return [(idx, merged_scores[idx]) for idx in ranked[:max(k, pool)]]


def rerank_with_cross_encoder(query, candidates, k=TOP_K_FACTUAL):
    """Rerank child chunks with cross-encoder."""
    if not candidates or cross_encoder is None:
        return candidates[:k]
    pairs = [[query, all_chunks[idx]] for idx, score in candidates]
    ce_scores = cross_encoder.predict(pairs)
    reranked = sorted(
        [(idx, float(ce_score)) for (idx, base_score), ce_score in zip(candidates, ce_scores)],
        key=lambda x: x[1],
        reverse=True
    )
    return reranked[:k]


def retrieve_and_swap_to_parents(query, k=TOP_K_FACTUAL, pool=POOL_K):
    """
    KEY STEP: Retrieve child chunks -> swap for parent chunks.
    This ensures the LLM receives full section context.
    """
    candidates = retrieve_multi_query(query, k=k, pool=pool)[:pool]
    ranked_children = rerank_with_cross_encoder(query, candidates, k=k)

    # SWAP: Replace child chunks with their parent chunks
    parent_chunks_to_return = []
    for child_idx, ce_score in ranked_children:
        child_metadata = chunk_metadata.get(child_idx, {})
        parent_id = child_metadata.get('parent_id')
        
        if parent_id and parent_id in parent_chunks_dict:
            parent_chunk = parent_chunks_dict[parent_id]
            parent_chunks_to_return.append({
                'text': parent_chunk['text'],
                'score': ce_score,
                'chunk_id': parent_chunk['id'],
                'source': parent_chunk['source'],
                'header': parent_chunk['header'],
            })

    # Remove duplicates, keep highest score
    unique_parents = {}
    for parent in parent_chunks_to_return:
        if parent['chunk_id'] not in unique_parents or parent['score'] > unique_parents[parent['chunk_id']]['score']:
            unique_parents[parent['chunk_id']] = parent

    return sorted(unique_parents.values(), key=lambda x: x['score'], reverse=True)[:k]


print("Retrieval pipeline ready with parent swapping.")


## 9. Utility Functions


In [ ]:
def safe_parse_json(text, fallback=None):
    """Parse JSON with markdown fence fallback."""
    try:
        return json.loads(text)
    except Exception:
        pass
    try:
        match = re.search(r'(\[.*\]|\{.*\})', text, re.DOTALL)
        if match:
            return json.loads(match.group(1))
    except Exception:
        pass
    return fallback


def sentence_split(text):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def readability_summary(text):
    try:
        grade = round(textstat.flesch_kincaid_grade(text), 2)
        reading_ease = round(textstat.flesch_reading_ease(text), 2)
        sentences = sentence_split(text)
        sentence_lengths = [len(s.split()) for s in sentences] or [len(text.split())]
        avg_sentence_length = round(float(np.mean(sentence_lengths)), 2)
        return {
            "grade_level": grade,
            "reading_ease": reading_ease,
            "avg_sentence_length": avg_sentence_length,
        }
    except Exception:
        return {"grade_level": 99, "reading_ease": 0, "avg_sentence_length": 99}


def clamp01(value):
    if value is None:
        return None
    try:
        return max(0.0, min(1.0, float(value)))
    except Exception:
        return None


INSURANCE_FILTER_TERMS = {
    "assured", "benefit", "claim", "coverage", "deferment", "exclusion",
    "income", "maturity", "policy", "premium", "rider", "settlement",
    "surrender", "term", "vesting"
}

def jargon_metrics(answer):
    answer_lower = answer.lower()
    matches = []
    for term in INSURANCE_FILTER_TERMS:
        pattern = r"\b" + re.escape(term.lower()) + r"\b"
        count = len(re.findall(pattern, answer_lower))
        if count:
            matches.extend([term] * count)
    total_words = max(len(re.findall(r"\b\w+\b", answer)), 1)
    jargon_count = len(matches)
    jargon_density = jargon_count / total_words
    jargon_free_score = max(0, min(1, 1 - (jargon_density * 3)))
    return {
        "jargon_count": jargon_count,
        "jargon_free_score": round(jargon_free_score, 4),
    }


print("Utility functions loaded.")


## 10. Advanced LLM-as-Judge Evaluation Metrics

### Methods Implemented:
1. **NLI-Based Claim Deconstruction** - Break answer into micro-claims
2. **Chain-of-Thought (CoT) Reasoning** - Judge reasons before scoring
3. **Citation & Source Correctness Audit** - Multi-step DAG-based verification


In [ ]:
GEMINI_MAX_RETRIES = 2
GEMINI_CALL_DELAY_SECONDS = 4
STOP_ON_GEMINI_QUOTA_EXCEEDED = True
GEMINI_QUOTA_EXCEEDED = False


def call_gemini(prompt):
    """Call Gemini with retry logic."""
    global GEMINI_QUOTA_EXCEEDED

    if GEMINI_QUOTA_EXCEEDED and STOP_ON_GEMINI_QUOTA_EXCEEDED:
        return None

    for attempt in range(GEMINI_MAX_RETRIES):
        try:
            if GEMINI_CALL_DELAY_SECONDS > 0:
                time.sleep(GEMINI_CALL_DELAY_SECONDS)
            response = gemini_model.generate_content(prompt)
            return response.text.strip()
        except Exception as e:
            message = str(e).lower()
            if "quota exceeded" in message:
                GEMINI_QUOTA_EXCEEDED = True
                print("Gemini quota exceeded.")
                return None
            if attempt < GEMINI_MAX_RETRIES - 1:
                print(f"Retry attempt {attempt + 1}...")
                time.sleep(5)
                continue
            print(f"Gemini call failed: {message}")
            return None
    return None


def evaluate_with_nli_and_cot(question, answer, context):
    """
    🥇 NLI-Based Claim Deconstruction with Chain-of-Thought
    
    Step 1: Break answer into micro-claims
    Step 2: Evaluate each against context using NLI (Entailed/Contradicted/Neutral)
    Step 3: Use CoT - Judge writes reasoning FIRST, then outputs scores
    
    Output: Faithfulness, Hallucination, Relevance, Coverage, ELI15, Citation Correctness
    """
    prompt = f"""
You are an expert RAG evaluation judge for insurance policies. You MUST use Chain-of-Thought reasoning.

CRITICAL INSTRUCTION: Write your full reasoning analysis FIRST (step-by-step), THEN output scores.
Do NOT output scores first and reason later.

QUESTION:
{question}

RETRIEVED CONTEXT (Insurance Policy Chunks):
{context}

GENERATED ANSWER:
{answer}

=== STEP 1: MICRO-CLAIM DECONSTRUCTION ===
Break the answer into atomic claims (e.g., "Plan covers water damage", "Maximum amount is $5000").
For each claim, determine:
- ENTAILED: Explicitly supported in context with exact/synonymous wording
- CONTRADICTED: Context says the opposite
- NEUTRAL/UNSUPPORTED: Not found in context

=== STEP 2: CHAIN-OF-THOUGHT ANALYSIS ===
Write 4-6 sentences analyzing:
1. What factual claims does the answer make?
2. How many are supported by the context? List any unsupported claims.
3. Are there any contradictions or invented numbers/riders?
4. Does the answer fully address the question?
5. Would a 15-year-old understand the language (considering jargon, sentence complexity)?
6. Are all citations/references traceable to specific context chunks?

=== STEP 3: NLI-BASED SCORING ===
Using your micro-claims analysis, score ONLY:

FAITHFULNESS: (entailed_claims + 0.5 * neutral_claims) / total_claims. If no claims, use 0.
HALLUCINATION: Fraction of contradicted or invented content (0=none, 1=mostly hallucinated).
RELEVANCE: Does answer address the question? (0-1)
COVERAGE: How much important context is used? (0-1)
ELI15_SCORE: Can a 15-year-old understand it? (0=very complex, 1=very simple)
CITATION_CORRECTNESS: Are all claims traceable to specific chunks? (0-1)

=== OUTPUT FORMAT ===
WRITE YOUR REASONING FIRST:
[Your 4-6 sentence analysis here]

THEN OUTPUT ONLY THIS JSON:
{{
  "faithfulness": 0.0,
  "hallucination": 0.0,
  "relevance": 0.0,
  "coverage": 0.0,
  "eli15_score": 0.0,
  "citation_correctness": 0.0,
  "reason": "[Your CoT reasoning - concise but complete]"
}}
"""
    raw = call_gemini(prompt)
    
    if not raw:
        return None

    # Extract JSON from CoT output
    result = safe_parse_json(raw, fallback=None)

    if result is None:
        return None
    
    # Clamp all scores to [0, 1]
    for key in ["faithfulness", "hallucination", "relevance", "coverage", "eli15_score", "citation_correctness"]:
        result[key] = clamp01(result.get(key))

    return result


def evaluate_citation_audit(question, answer, context):
    """
    🥇 Multi-Step Citation Audit (G-Eval DAG-Based)
    
    Verify that every claim in the answer is backed by exact chunk references.
    """
    prompt = f"""
You are a Citation Auditor for insurance policy evaluation.

QUESTION:
{question}

GENERATED ANSWER:
{answer}

RETRIEVED CONTEXT CHUNKS:
{context}

=== CITATION AUDIT CHECKLIST ===
1. Identify every factual claim (numbers, exclusions, conditions, riders, etc.)
2. For each claim, verify:
   - Is it found in the context chunks?
   - What is the exact text from context that supports it?
   - Is it paraphrased or directly quoted?
3. Flag claims that:
   - Are completely unsupported
   - Invert the meaning from context (e.g., says "covered" when context says "excluded")
   - Cite wrong numbers or facts

WRITE YOUR AUDIT:
[List each claim, its context support, and any discrepancies]

THEN OUTPUT ONLY THIS JSON:
{{
  "citation_correctness": 0.0,
  "supported_claims": [],
  "unsupported_claims": [],
  "inverted_claims": [],
  "audit_reason": "[Your audit findings]"
}}
"""
    raw = call_gemini(prompt)
    
    if not raw:
        return {"citation_correctness": None}

    result = safe_parse_json(raw, fallback={})
    result["citation_correctness"] = clamp01(result.get("citation_correctness"))

    return result


print("Advanced LLM-as-Judge evaluation loaded.")


## 11. Run Full Evaluation Pipeline


In [ ]:
results = []

LOCAL_TEST_QUESTION_LIMIT = 15
rows_to_evaluate = min(LOCAL_TEST_QUESTION_LIMIT, len(df_qa))

for question_number in range(1, rows_to_evaluate + 1):
    print(f"\n{'='*80}")
    print(f"Question {question_number}/{rows_to_evaluate}")
    print(f"{'='*80}")

    if GEMINI_QUOTA_EXCEEDED and STOP_ON_GEMINI_QUOTA_EXCEEDED:
        print("Stopping: Gemini quota exhausted.")
        break

    row = df_qa.iloc[question_number - 1]
    question = row["Question"]
    answer = row["LLM_Output"]

    print(f"Question: {question[:80]}...")

    # Readability & jargon
    readability_info = readability_summary(answer)
    jargon_info = jargon_metrics(answer)

    # Retrieve and swap child chunks to parent chunks
    print("Retrieving parent chunks...")
    parent_chunks_retrieved = retrieve_and_swap_to_parents(
        question,
        k=TOP_K_FACTUAL,
        pool=POOL_K
    )

    if not parent_chunks_retrieved:
        factual_context = "No context available."
        print("  No relevant parent chunks found.")
    else:
        factual_context = "\n\n---\n\n".join([p['text'] for p in parent_chunks_retrieved])
        print(f"  Retrieved {len(parent_chunks_retrieved)} parent chunk(s)")

    # 🥇 NLI-Based Evaluation with Chain-of-Thought
    print("Running NLI-based evaluation with CoT...")
    evaluation = evaluate_with_nli_and_cot(question, answer, factual_context)

    if GEMINI_QUOTA_EXCEEDED and STOP_ON_GEMINI_QUOTA_EXCEEDED:
        print("Stopping: Gemini quota exhausted.")
        break

    if evaluation is None:
        print("Evaluation failed. Skipping...")
        continue

    # Extract scores
    faithfulness = clamp01(evaluation.get("faithfulness"))
    hallucination = clamp01(evaluation.get("hallucination"))
    relevance = clamp01(evaluation.get("relevance"))
    coverage = clamp01(evaluation.get("coverage"))
    eli15_score = clamp01(evaluation.get("eli15_score"))
    citation_correctness = clamp01(evaluation.get("citation_correctness"))
    jargon_free = clamp01(jargon_info.get("jargon_free_score"))
    reason = evaluation.get("reason", "")

    print(f"Scores: F={faithfulness} H={hallucination} R={relevance} C={coverage} E={eli15_score} Cit={citation_correctness} J={jargon_free}")

    results.append({
        "Q#": question_number,
        "Question": question[:100],
        "Faithfulness": faithfulness,
        "Hallucination": hallucination,
        "Relevance": relevance,
        "Coverage": coverage,
        "ELI15_Score": eli15_score,
        "Jargon_Free_Score": jargon_free,
        "Citation_Correctness": citation_correctness,
        "Reason": reason
    })

df_results = pd.DataFrame(results)

print(f"\n{'='*80}")
print(f"Evaluation Complete: {len(df_results)} questions evaluated")
print(f"{'='*80}")


## 12. Export Results to Excel


In [ ]:
# Save to Excel
excel_file = "evaluation_results_advanced_metrics.xlsx"

df_results.to_excel(excel_file, index=False, sheet_name="Evaluation")

# Format
wb = openpyxl.load_workbook(excel_file)
ws = wb["Evaluation"]

# Format Reason column with text wrapping
for cell in ws[1]:
    if cell.value == "Reason":
        reason_col = cell.column
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=reason_col).alignment = Alignment(wrap_text=True, vertical="top")
        ws.column_dimensions[get_column_letter(reason_col)].width = 100
        break

wb.save(excel_file)

print(f"\nResults saved to: {excel_file}")
print(f"\nFinal Results Summary:")
print(df_results[['Q#', 'Faithfulness', 'Hallucination', 'Relevance', 'Coverage', 'ELI15_Score', 'Citation_Correctness']].to_string())

# Download
try:
    files.download(excel_file)
    print(f"\n✅ File downloaded: {excel_file}")
except Exception as e:
    print(f"Download error: {e}")
